<a href="https://colab.research.google.com/github/ZoryAce/Topicos/blob/main/ENTREGA2/TALLER6/E6_NeuralNetworksPyTorchNLP_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Excercise 6**
## NLP with Pytorch 🔥

**INTEGRANTES:**

Luis Alejandro Garzón Ramirez

Zorayda Acevedo Jimenez

Yulieth Danitza Aguillón Ortega




Use keras framework to solve the below exercises.


In [1]:
#import numpy as np
import keras
import pandas as pd
import matplotlib.pyplot as plt

## 6.1 Predict rating of a movie using Pytorch

**Exercise:** Use keras framework to predict rating.

In [2]:
dataTraining = pd.read_csv('https://github.com/sergiomora03/AdvancedTopicsAnalytics/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)

In [3]:
plots = dataTraining['plot']
y = (dataTraining['rating'] >= dataTraining['rating'].mean()).astype(int)

In [4]:
plots

,plot
3107,most is the story of a single father who takes...
900,a serial killer decides to teach the secrets o...
6724,"in sweden , a female blackmailer with a disfi..."
4704,"in a friday afternoon in new york , the presi..."
2582,"in los angeles , the editor of a publishing h..."
...,...
8417,""" our marriage , their wedding . "" it ' s l..."
1592,"the wandering barbarian , conan , alongside ..."
1723,"like a tale spun by scheherazade , kismet fol..."
7605,"mrs . brisby , a widowed mouse , lives in a..."


In [5]:
y

,rating
3107,1
900,0
6724,1
4704,1
2582,1
...,...
8417,0
1592,0
1723,0
7605,1


## Data Precosessing

- Remove stopwords
- Lowercase
- split the text in words
- pad_sequences

El codigo a continuación realiza lo siguiente:
1. Importación de librerías: Se usan PyTorch, sklearn y nltk para manejar datos, modelos y procesamiento de texto.

2. Carga de datos: Se descarga un dataset desde un enlace (dataTraining.csv), que contiene reseñas de películas (plot) y sus calificaciones (rating).

3. Creación de variable objetivo (y): Se convierte la calificación en una variable binaria:

1 → si la calificación es mayor o igual al promedio.

0 → si es menor.

4. Preprocesamiento del texto: Convierte el texto a minúsculas, elimina caracteres especiales y remueve palabras vacías (stopwords en inglés). Usa frecuencia de palabras (Counter) para construir un vocabulario limitado a 8000 palabras más comunes.

5. Conversión del texto en secuencias de números: Cada palabra se reemplaza por su índice en el vocabulario. Se recorta o rellena las secuencias a una longitud fija (150 palabras).

6. Conversión de datos a tensores de PyTorch: Se convierte el texto procesado (X_padded) y las etiquetas (y) en tensores de PyTorch.

7.  División en entrenamiento y prueba: Se separan los datos en 80% para entrenar y 20% para prueba usando train_test_split.

8.  Creación de DataLoaders: Se organizan los datos en lotes (batch_size=32) para ser procesados eficientemente por el modelo en PyTorch.


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
import nltk
from nltk.corpus import stopwords
from collections import Counter
import re

nltk.download('stopwords')
dataTraining = pd.read_csv('https://github.com/sergiomora03/AdvancedTopicsAnalytics/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
plots = dataTraining['plot']
y = (dataTraining['rating'] >= dataTraining['rating'].mean()).astype(int)

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return words

processed_plots = plots.apply(preprocess_text)
word_counts = Counter(word for plot in processed_plots for word in plot)
vocab_size = 8000
vocab = {word: i+1 for i, (word, _) in enumerate(word_counts.most_common(vocab_size))}

def text_to_sequence(text, vocab):
    return [vocab.get(word, 0) for word in text]

sequences = [text_to_sequence(plot, vocab) for plot in processed_plots]
max_length = 150
X_padded = [seq[:max_length] + [0]*(max_length - len(seq)) if len(seq) < max_length else seq[:max_length] for seq in sequences]

X_padded = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y.values, dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(X_padded, y_tensor, test_size=0.2, random_state=42)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Build Model

Create a neural network to predict the rating of a movie, calculate the testing set accuracy.

Este código define e inicializa un modelo de Deep Learning con GRU (Gated Recurrent Unit) para predecir si una película tiene una calificación alta o baja a partir de su sinopsis.

**1. Definición del modelo ImprovedGRUMovieRatingPredictor:** Es una clase que hereda de torch.nn.Module, lo que significa que es un modelo en PyTorch.

Usa una capa de embedding para convertir palabras en vectores numéricos (nn.Embedding).

Usa una capa GRU (nn.GRU) con 3 capas (num_layers=3) para procesar las secuencias de texto.

Usa Dropout (0.6) para reducir el sobreajuste.

Tiene dos capas completamente conectadas (fc1, fc2) para transformar la salida de la GRU en una probabilidad.

Usa la función sigmoide (nn.Sigmoid) para obtener una salida entre 0 y 1 (ideal para clasificación binaria).

**2. Parámetros del modelo:**

Tamaño del embedding: 256 (mayor capacidad de representación de palabras).

Tamaño de la capa oculta (hidden_size): 256 (más neuronas para aprender patrones).

Número de capas GRU: 3 (modelo más profundo).

Dropout: 0.6 (prevención de sobreajuste).

**3. Inicialización del modelo:**  Se crea una instancia del modelo ImprovedGRUMovieRatingPredictor con los parámetros definidos.

In [7]:
class ImprovedGRUMovieRatingPredictor(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size, num_layers, dropout):
        super(ImprovedGRUMovieRatingPredictor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.gru(x)
        x = self.dropout(x[:, -1, :])
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return self.sigmoid(x)

# Parámetros del modelo
embed_size = 256  # Aumentar el tamaño del embedding
hidden_size = 256  # Aumentar el tamaño de la capa oculta
output_size = 1
num_layers = 3  # Agregar más capas ocultas
dropout = 0.6  # Aumentar el Dropout

# Inicializar el modelo
model = ImprovedGRUMovieRatingPredictor(vocab_size+1, embed_size, hidden_size, output_size, num_layers, dropout)

Este código define y ejecuta el entrenamiento de un modelo de red neuronal en PyTorch usando la función de pérdida BCELoss y el optimizador Adam.

In [8]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

def train_model(model, train_loader, criterion, optimizer, epochs=20):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}')

train_model(model, train_loader, criterion, optimizer, epochs=20)

Epoch [1/20], Loss: 0.6964
Epoch [2/20], Loss: 0.6942
Epoch [3/20], Loss: 0.6896
Epoch [4/20], Loss: 0.6706
Epoch [5/20], Loss: 0.6112
Epoch [6/20], Loss: 0.5144
Epoch [7/20], Loss: 0.3796
Epoch [8/20], Loss: 0.2413
Epoch [9/20], Loss: 0.1503
Epoch [10/20], Loss: 0.1041
Epoch [11/20], Loss: 0.0703
Epoch [12/20], Loss: 0.0619
Epoch [13/20], Loss: 0.0434
Epoch [14/20], Loss: 0.0364
Epoch [15/20], Loss: 0.0386
Epoch [16/20], Loss: 0.0379
Epoch [17/20], Loss: 0.0314
Epoch [18/20], Loss: 0.0247
Epoch [19/20], Loss: 0.0368
Epoch [20/20], Loss: 0.0277


Este código evalúa el rendimiento de un modelo de clasificación binaria entrenado en PyTorch, calculando su precisión (accuracy) en un conjunto de prueba (test_loader).

In [9]:
def evaluate_model(model, test_loader):
    model.eval()
    y_pred = []
    y_true = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            y_pred.extend(outputs.squeeze().numpy())
            y_true.extend(labels.numpy())
    y_pred = [1 if pred >= 0.5 else 0 for pred in y_pred]
    accuracy = accuracy_score(y_true, y_pred)
    return accuracy

accuracy = evaluate_model(model, test_loader)
print(f'Accuracy on the test set: {accuracy:.4f}')

Accuracy on the test set: 0.5598


Observamos que el resultado de AUC en test es de 0.5598, por lo que se decide intentar con otro modelo.

# **Otra opción**

Para buscar un mejor resultado de AUC

In [10]:
!pip install --upgrade --force-reinstall numpy==1.26.4 scipy==1.13.1
!pip install --upgrade --force-reinstall gensim

!pip install gensim

!pip uninstall -y nltk
!pip install nltk


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 7.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 11.5 MB/s eta 0:00:00
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Us

ERROR: Operation cancelled by user
^C
^C
^C


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import nltk
import re
import gensim.downloader as api
import pandas as pd
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
# 🔹 Cargar datos
dataTraining = pd.read_csv('https://github.com/sergiomora03/AdvancedTopicsAnalytics/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
plots = dataTraining['plot']
y = (dataTraining['rating'] >= dataTraining['rating'].mean()).astype(int)

stop_words = set(stopwords.words('english'))

# 🔹 Función de preprocesamiento mejorado
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Eliminar caracteres especiales
    words = word_tokenize(text)  # Tokenización avanzada
    words = [word for word in words if word not in stop_words]  # Eliminar stopwords
    return words

# Aplicar preprocesamiento a las reseñas
processed_plots = plots.apply(preprocess_text)

# Construir vocabulario con las palabras más comunes
vocab_size = 8000
word_counts = Counter(word for plot in processed_plots for word in plot)
vocab = {word: i+1 for i, (word, _) in enumerate(word_counts.most_common(vocab_size))}

# 🔹 Convertir texto a secuencias numéricas
def text_to_sequence(text, vocab):
    return [vocab.get(word, 0) for word in text]

sequences = [text_to_sequence(plot, vocab) for plot in processed_plots]

# Padding para que todas las secuencias tengan la misma longitud
max_length = 150
X_padded = [seq[:max_length] + [0]*(max_length - len(seq)) if len(seq) < max_length else seq[:max_length] for seq in sequences]

# Convertir a tensores
X_padded = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y.values, dtype=torch.float32)

# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_padded, y_tensor, test_size=0.2, random_state=42)

# Crear dataloaders
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 🔹 Descargar embeddings pre-entrenados (GloVe 100D)
word_vectors = api.load("glove-wiki-gigaword-100")

# Crear matriz de embeddings
embedding_dim = 100
embedding_matrix = torch.zeros((vocab_size+1, embedding_dim))
for word, idx in vocab.items():
    if word in word_vectors:
        embedding_matrix[idx] = torch.tensor(word_vectors[word])

# 🔹 Definir Capa de Atención
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, x):
        attn_weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * attn_weights, dim=1)

# 🔹 Definir Modelo Mejorado con GRU Bidireccional y Atención
class ImprovedGRUMovieRatingPredictor(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)  # Usar embeddings pre-entrenados
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.attention = Attention(hidden_size * 2)  # *2 porque es bidireccional
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.gru(x)
        x = self.attention(x)  # Aplicar atención
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return self.sigmoid(x)

# 🔹 Inicializar Modelo Mejorado
embed_size = 100  # Tamaño de embedding (de GloVe)
hidden_size = 256
output_size = 1
num_layers = 3
dropout = 0.6

model = ImprovedGRUMovieRatingPredictor(vocab_size+1, embed_size, hidden_size, output_size, num_layers, dropout)

# 🔹 Configurar Entrenamiento
criterion = nn.BCELoss()  # Binary Cross-Entropy para clasificación binaria
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 🔹 Entrenamiento del Modelo
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        X_batch, y_batch = batch
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        predictions = model(X_batch).squeeze()
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")

# 🔹 Evaluación del Modelo
model.eval()
y_pred, y_true = [], []

with torch.no_grad():
    for batch in test_loader:
        X_batch, y_batch = batch
        X_batch = X_batch.to(device)

        predictions = model(X_batch).squeeze()
        y_pred.extend(predictions.cpu().numpy())
        y_true.extend(y_batch.numpy())

# Convertir predicciones a 0 o 1
y_pred = [1 if pred >= 0.5 else 0 for pred in y_pred]

# Calcular precisión
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Epoch 1/5, Loss: 0.6730
Epoch 2/5, Loss: 0.6202
Epoch 3/5, Loss: 0.5490
Epoch 4/5, Loss: 0.4482
Epoch 5/5, Loss: 0.3488
Test Accuracy: 0.6111


Se observa que este segundo modelo obtuvo un mejor resutlado de AUC: 0.6111